# [9665] BERTopic
Data file:
* https://raw.githubusercontent.com/vjavaly/Baruch-CIS-9665/main/data/TeePublic_reviews_10k.csv

This extensive dataset, comprised of over 250,000 customer reviews, offers a detailed exploration of customer experiences on TeePublic, an online platform renowned for its diverse collection of fashion items. The dataset spans crucial information, including reviewer_id, store_location, latitude, longitude, date, month, year, title, review, and the review-label indicating a rating on a scale of 1 to 5.
  * reviewer_id: A unique identifier for each reviewer, ensuring anonymity and privacy.
  * store_location: Geographic information specifying the location of the TeePublic fashion store.
  * latitude: The latitude coordinate of the store's location, providing precise geospatial data.
  * longitude: The longitude coordinate of the store's location, offering detailed geographic insights.
  * date: The specific date when the review was posted, enabling temporal analysis.
  * month: The month in which the review was posted, facilitating monthly trends exploration.
  * year: The year of the review, allowing for yearly analysis and trend identification.
  * title: The title associated with each review, capturing succinct sentiments or key points.
  * review: The textual content of the review, presenting detailed feedback from customers.
  * review-label: The reviewer's rating on a scale from 1 to 5, providing a quantitative measure of satisfaction.

NOTE: Reduced to 10,000 reviews by professor

### Citation
```
@article{grootendorst2022bertopic,
  title={BERTopic: Neural topic modeling with a class-based TF-IDF procedure},
  author={Grootendorst, Maarten},
  journal={arXiv preprint arXiv:2203.05794},
  year={2022}
}
```

In [1]:
from datetime import datetime
print(f'Run time: {datetime.now().strftime("%D %T")}')

Run time: 04/18/25 15:49:05


### Import libraries

In [2]:
%%time

! pip install bertopic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.6/150.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 57.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalli

In [3]:
import pandas as pd
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from bertopic import BERTopic

In [4]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

### Load data

In [5]:
df = pd.read_csv('https://raw.githubusercontent.com/vjavaly/Baruch-CIS-9665/main/data/TeePublic_reviews_10k.csv')
df.shape

(10000, 10)

### Examine data

In [6]:
df.head()

,reviewer_id,store_location,latitude,longitude,date,month,year,title,review,review-label
0,114649.0,US,37.09024,-95.712891,2022,2,2011 00:00:00,Love the magnets,"Love the magnets! They are thin but strong, an...",5
1,29288.0,US,37.09024,-95.712891,2022,12,2030 00:00:00,The process was simple and easy,"The process was simple and easy, shipping was ...",5
2,70483.0,US,37.09024,-95.712891,2022,8,2017 00:00:00,Account banned for no reason,I had my account banned for no good reason at ...,1
3,7180.0,US,37.09024,-95.712891,2023,5,2021 00:00:00,Excellent customer service,Excellent customer service. Quick and easy exc...,5
4,155554.0,US,37.09024,-95.712891,2021,4,2026 00:00:00,The shirts were awesome,The shirts were awesome! My sisters love the f...,5


### Prepare data

In [7]:
# Drop rows with missing values
df = df.dropna().reset_index(drop=True)
df.shape

(8887, 10)

In [8]:
df['combined_text'] = df['title'] + ' ' + df['review']
df.head()

,reviewer_id,store_location,latitude,longitude,date,month,year,title,review,review-label,combined_text
0,114649.0,US,37.09024,-95.712891,2022,2,2011 00:00:00,Love the magnets,"Love the magnets! They are thin but strong, an...",5,Love the magnets Love the magnets! They are th...
1,29288.0,US,37.09024,-95.712891,2022,12,2030 00:00:00,The process was simple and easy,"The process was simple and easy, shipping was ...",5,The process was simple and easy The process wa...
2,70483.0,US,37.09024,-95.712891,2022,8,2017 00:00:00,Account banned for no reason,I had my account banned for no good reason at ...,1,Account banned for no reason I had my account ...
3,7180.0,US,37.09024,-95.712891,2023,5,2021 00:00:00,Excellent customer service,Excellent customer service. Quick and easy exc...,5,Excellent customer service Excellent customer ...
4,155554.0,US,37.09024,-95.712891,2021,4,2026 00:00:00,The shirts were awesome,The shirts were awesome! My sisters love the f...,5,The shirts were awesome The shirts were awesom...


In [9]:
# Create function to clean text
lem = WordNetLemmatizer()
stop = set(stopwords.words('english'))
punct = string.punctuation

# Create function to clean_text
def clean_text(text):
    # print(f'{type(text)} : {text}')
    text = re.sub(r'\s+', ' ', text).translate(str.maketrans('', '', punct)).lower()
    tokens = text.split()
    tokens = [lem.lemmatize(word) for word in tokens if word not in stop]
    if len(tokens) > 2:
        return ' '.join(tokens)

# Perform necessary text preprocessing

In [10]:
%%time

# Clean combined_text column
df['clean_combined_text'] = df['combined_text'].apply(clean_text)

CPU times: user 7.59 s, sys: 237 ms, total: 7.83 s
Wall time: 11.9 s


In [11]:
# Drop rows with missing values
df = df.dropna().reset_index(drop=True)
df.shape

(8876, 12)

### Use BERTopic pretrained model

In [12]:
# To use the default model (all-MiniLM-L6-v2), you do not have to specify a model name
topic_model = BERTopic()

In [13]:
# Display default hyperparameters for model BERTopic
topic_model.get_params()

{'calculate_probabilities': False,
 'ctfidf_model': ClassTfidfTransformer(),
 'embedding_model': None,
 'hdbscan_model': HDBSCAN(min_cluster_size=10, prediction_data=True),
 'language': 'english',
 'low_memory': False,
 'min_topic_size': 10,
 'n_gram_range': (1, 1),
 'nr_topics': None,
 'representation_model': None,
 'seed_topic_list': None,
 'top_n_words': 10,
 'umap_model': UMAP(low_memory=False, metric='cosine', min_dist=0.0, n_components=5),
 'vectorizer_model': CountVectorizer(),
 'verbose': False,
 'zeroshot_min_similarity': 0.7,
 'zeroshot_topic_list': None}

In [14]:
%%time

# Train the BERTopic model on the cleaned combined text
topics, probs = topic_model.fit_transform(df['clean_combined_text'])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

CPU times: user 5min 46s, sys: 6.53 s, total: 5min 52s
Wall time: 5min 50s


In [15]:
# Display topics
#  Topic_ID -1 : refers to outlier documents that do not fit well into any topics
#  Topic_ID 0 : groups together documents that have little meaning
#  Topic_ID 1 : the most frequent topic in the corpus
#  Topic_ID 2 and onwards : the 2nd, …, Nth most frequent topic in the corpus

topic_model.get_topics()

{-1: [('shirt', np.float64(0.010129930123621407)),
  ('quality', np.float64(0.009575383954318995)),
  ('design', np.float64(0.009184749954005856)),
  ('great', np.float64(0.008937834954504214)),
  ('good', np.float64(0.008163928227451841)),
  ('love', np.float64(0.007901653627270704)),
  ('tshirt', np.float64(0.007734781755366276)),
  ('order', np.float64(0.007538997333784741)),
  ('ordered', np.float64(0.0073776618157323295)),
  ('product', np.float64(0.007248034433018071))],
 0: [('size', np.float64(0.025315254864392657)),
  ('small', np.float64(0.023912309383887116)),
  ('shirt', np.float64(0.015024834175313674)),
  ('xl', np.float64(0.01444808498188088)),
  ('large', np.float64(0.013789504084448692)),
  ('ordered', np.float64(0.013581294557044226)),
  ('run', np.float64(0.013174788114769074)),
  ('medium', np.float64(0.012872629112101219)),
  ('sizing', np.float64(0.012486667654051587)),
  ('wrong', np.float64(0.012157339637221033))],
 1: [('shirt', np.float64(0.027387435739073776)

In [16]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,3855,-1_shirt_quality_design_great,"[shirt, quality, design, great, good, love, ts...",[service great got myï¿½ï¿½ï¿½ service great g...
1,0,816,0_size_small_shirt_xl,"[size, small, shirt, xl, large, ordered, run, ...",[shirt run small shirt run small menï¿½ï¿½ï¿½ï...
2,1,494,1_shirt_loved_love_happy,"[shirt, loved, love, happy, arrived, ordered, ...",[happy shirt iï¿½ï¿½ï¿½ happy shirt bought gre...
3,2,273,2_exactly_item_order_product,"[exactly, item, order, product, arrived, time,...",[order arrived time iï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ ...
4,3,201,3_hoodie_hoodies_hoody_son,"[hoodie, hoodies, hoody, son, love, fit, grand...",[love new hoodie love new hoodie definitely or...
...,...,...,...,...,...
85,84,12,84_issue_peep_service_customer,"[issue, peep, service, customer, corrected, fi...",[service great service great issue order custo...
86,85,12,85_teepublic_recommend_always_experience,"[teepublic, recommend, always, experience, bla...",[true review last purchase teepublic usual gre...
87,86,11,86_early_condition_came_arrived,"[early, condition, came, arrived, everything, ...",[received good condition timely received good ...
88,87,11,87_teepublic_from_second_item,"[teepublic, from, second, item, first, time, b...",[teepublic quick let know thatï¿½ï¿½ï¿½ teepub...


In [17]:
# Display topic sizes
#  topic_ID : # of documents assigned to topic
topic_model.topic_sizes_

Counter({61: 19,
         31: 42,
         72: 15,
         12: 74,
         -1: 3855,
         2: 273,
         0: 816,
         40: 33,
         20: 52,
         34: 39,
         6: 120,
         38: 37,
         7: 116,
         4: 161,
         58: 21,
         79: 13,
         29: 43,
         1: 494,
         11: 75,
         27: 44,
         18: 56,
         75: 14,
         28: 43,
         35: 38,
         74: 14,
         26: 44,
         70: 15,
         69: 15,
         9: 93,
         3: 201,
         48: 30,
         8: 116,
         50: 27,
         46: 32,
         57: 22,
         47: 30,
         55: 24,
         10: 83,
         19: 54,
         33: 39,
         24: 46,
         37: 38,
         52: 25,
         5: 145,
         41: 33,
         60: 19,
         42: 33,
         59: 20,
         22: 48,
         76: 14,
         23: 48,
         68: 17,
         45: 32,
         14: 63,
         17: 56,
         25: 45,
         21: 49,
         32: 42,
         88: 

In [18]:
# Display list of words and their weights that comprise Topic 1
topic_model.get_topic(1)

[('shirt', np.float64(0.027387435739073776)),
 ('loved', np.float64(0.014917872944211602)),
 ('love', np.float64(0.013548031268732106)),
 ('happy', np.float64(0.013079362948142033)),
 ('arrived', np.float64(0.012086411428353752)),
 ('ordered', np.float64(0.01075253401727201)),
 ('and', np.float64(0.010636221380868203)),
 ('quality', np.float64(0.010570887691528356)),
 ('came', np.float64(0.010105672474757004)),
 ('good', np.float64(0.00916046167881006))]

In [19]:
# Display the embeddings for Topic 1
topic_model.topic_embeddings_[1]

array([-1.25511345e-02,  7.64826685e-02,  3.94381732e-02,  2.76250448e-02,
        2.75072977e-02, -3.97611149e-02,  5.45203174e-03,  1.09360022e-02,
       -2.50674412e-02,  7.28495093e-03,  4.74162251e-02, -3.23916525e-02,
        2.31039617e-02, -1.88894924e-02, -6.29254524e-03, -9.05061793e-03,
        2.38782652e-02, -6.21269736e-03, -7.89026543e-02, -6.06893450e-02,
       -2.41863355e-02, -2.72534452e-02, -6.80497335e-03,  1.42028779e-02,
       -5.13012633e-02, -4.32405956e-02, -5.09779081e-02,  5.58213284e-03,
       -8.42171069e-03, -4.52680551e-02, -2.38900185e-02,  7.15687424e-02,
        7.22024143e-02,  2.82762516e-02,  4.32896614e-03, -4.23753746e-02,
        2.41754521e-02, -8.59301724e-03,  3.84099851e-03,  1.52550600e-02,
       -1.23068793e-02, -5.50342835e-02, -2.29429156e-02, -1.16157637e-03,
       -5.31992614e-02,  1.69201344e-02, -6.52516726e-03,  6.96344674e-02,
       -1.50323911e-02,  5.49124181e-02, -1.37709212e-02, -2.91082375e-02,
       -3.38751040e-02, -

In [20]:
topic_model.get_document_info(df['clean_combined_text'])

,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Probability,Representative_document
0,love magnet love magnet thin strong like desig...,61,61_magnet_car_coolbut_lsu,"[magnet, car, coolbut, lsu, sticker, fastlove,...",[car magnet ordered college mascot magnet car ...,magnet - car - coolbut - lsu - sticker - fastl...,1.000000,True
1,process simple easy process simple easy shippi...,31,31_easy_process_simple_order,"[easy, process, simple, order, shipping, shirt...",[process simple easy process simple easy shipp...,easy - process - simple - order - shipping - s...,1.000000,True
2,account banned reason account banned good reas...,72,72_account_term_explanation_banned,"[account, term, explanation, banned, legal, wi...",[alleged violation unspecified term caused acc...,account - term - explanation - banned - legal ...,1.000000,False
3,excellent customer service excellent customer ...,12,12_service_excellent_great_customer,"[service, excellent, great, customer, shirt, o...",[great shirt great shirt great customer servic...,service - excellent - great - customer - shirt...,1.000000,False
4,shirt awesome shirt awesome sister love funny ...,-1,-1_shirt_quality_design_great,"[shirt, quality, design, great, good, love, ts...",[service great got myï¿½ï¿½ï¿½ service great g...,shirt - quality - design - great - good - love...,0.000000,False
...,...,...,...,...,...,...,...,...
8871,happy quick arrival ofï¿½ï¿½ï¿½ happy quick ar...,-1,-1_shirt_quality_design_great,"[shirt, quality, design, great, good, love, ts...",[service great got myï¿½ï¿½ï¿½ service great g...,shirt - quality - design - great - good - love...,0.000000,False
8872,great quality great quality fast delivery use ...,36,36_delivery_quality_good_fast,"[delivery, quality, good, fast, excellent, qui...","[good quality good quality fast delivery, good...",delivery - quality - good - fast - excellent -...,0.979244,False
8873,contacted legal regarding removalï¿½ï¿½ï¿½ con...,72,72_account_term_explanation_banned,"[account, term, explanation, banned, legal, wi...",[alleged violation unspecified term caused acc...,account - term - explanation - banned - legal ...,0.971839,False
8874,good shirt fantastic design huge selection goo...,-1,-1_shirt_quality_design_great,"[shirt, quality, design, great, good, love, ts...",[service great got myï¿½ï¿½ï¿½ service great g...,shirt - quality - design - great - good - love...,0.000000,False


In [21]:
# Display the representative docs for Topic 1
topic_model.get_representative_docs(1)

['happy shirt iï¿½ï¿½ï¿½ happy shirt bought great quality love design',
 'ordered 6 shirt arrivedï¿½ï¿½ï¿½ ordered 6 shirt arrived quickly look great',
 'shirt arrived good shippingï¿½ï¿½ï¿½ shirt arrived good shipping time design looked great son loved definitely recommend']

In [22]:
# Generates labels for each topic in a used-defined format
topic_model.generate_topic_labels(nr_words=4, separator='+')

['-1+shirt+quality+design+great',
 '0+size+small+shirt+xl',
 '1+shirt+loved+love+happy',
 '2+exactly+item+order+product',
 '3+hoodie+hoodies+hoody+son',
 '4+package+lost+order+item',
 '5+mask+face+nose+made',
 '6+sweatshirt+hooded+crewneck+buffalo',
 '7+tee+public+great+soft',
 '8+tshirts+tshirt+happy+comfortable',
 '9+love+shirt+design+loved',
 '10+teepublic+always+selection+customer',
 '11+artist+artwork+art+independent',
 '12+service+excellent+great+customer',
 '13+fit+true+quality+size',
 '14+tshirt+tshirts+size+large',
 '15+delivery+fast+great+shirt',
 '16+mug+coffee+cup+broken',
 '17+customer+service+excellent+product',
 '18+teepublic+favorite+love+find',
 '19+size+wrong+replaced+ordered',
 '20+tee+ah+nephew+special',
 '21+soft+comfortable+super+shirt',
 '22+gift+christmas+husband+surprise',
 '23+tshirt+tshirts+service+arrived',
 '24+delivery+quick+fast+product',
 '25+service+excellent+good+great',
 '26+onesie+granddaughter+poo+im',
 '27+design+beautiful+amazing+unique',
 '28+shi

In [23]:
# Visualize word bar chart for top 10 topics
topic_model.visualize_barchart(top_n_topics=10, n_words=4)

### Reduce topics to top N topics

In [24]:
bertopic_top_N = topic_model.reduce_topics(df['clean_combined_text'], nr_topics=5)

In [25]:
# Display reduced topics
bertopic_top_N.get_topics()

{-1: [('shirt', np.float64(0.07814171658639454)),
  ('great', np.float64(0.05663114358356436)),
  ('quality', np.float64(0.05516101069402389)),
  ('design', np.float64(0.040611102002803254)),
  ('love', np.float64(0.03847898482167635)),
  ('good', np.float64(0.035968061733018114)),
  ('ordered', np.float64(0.0347631887923937)),
  ('order', np.float64(0.03399811808765492)),
  ('product', np.float64(0.02863277439556132)),
  ('tshirt', np.float64(0.028414248467295785))],
 0: [('shirt', np.float64(0.07627819531008875)),
  ('great', np.float64(0.06228847907011604)),
  ('quality', np.float64(0.05018436589191344)),
  ('service', np.float64(0.0436643195168264)),
  ('love', np.float64(0.04304706171817738)),
  ('ordered', np.float64(0.0420477415683622)),
  ('size', np.float64(0.04064214608195645)),
  ('order', np.float64(0.038053331863740096)),
  ('good', np.float64(0.03356432713721314)),
  ('customer', np.float64(0.03154806878178811))],
 1: [('magnet', np.float64(0.8065494822169752)),
  ('car',

In [26]:
# Visualize word bar chart for reduced topics
bertopic_top_N.visualize_barchart(n_words=4)